# Sorting

This section compares practical sorting approaches from baseline to efficient methods.

**Learning Goals**
- Use Python built-in sorting effectively
- Implement insertion, merge, bubble, and quicksort
- Compare runtime trends across input patterns
- Explain stability and memory trade-offs

In [1]:
from time import perf_counter
import random

## Python Built-in Sorting

Use `sorted(iterable)` to return a new list and `list.sort()` for in-place sorting.
Python's Timsort is stable and highly optimized for real workloads.

In [2]:
nums = [5, 2, 9, 1, 5, 6]
print(sorted(nums))
print(nums)

nums.sort()
print(nums)

[1, 2, 5, 5, 6, 9]
[5, 2, 9, 1, 5, 6]
[1, 2, 5, 5, 6, 9]


### Stable Sorting and `key=`

**Stability** means that when two elements compare as equal, their original relative order is preserved in the output. Python's `sorted()` and `.sort()` are both stable.

Stability matters when records share the same key and you want original order preserved.

In [3]:
students = [
    {"name": "Ava", "score": 90},
    {"name": "Ben", "score": 90},
    {"name": "Cara", "score": 85},
]

sorted_students = sorted(students, key=lambda s: s["score"], reverse=True)
for s in sorted_students:
    print(s)
# Ava and Ben share score=90; Ava appears first, confirming stable sort preserves original order.

{'name': 'Ava', 'score': 90}
{'name': 'Ben', 'score': 90}
{'name': 'Cara', 'score': 85}


## Core Sorting Algorithms

The following implementations are included for conceptual comparison and practice.

**In-place vs. out-of-place:** An in-place sort rearranges elements without allocating extra memory proportional to the input (bubble sort, insertion sort). An out-of-place sort returns a new list and leaves the original untouched — useful when you need both the original and the sorted version (merge sort, `sorted()`, and the quicksort below).

### Insertion Sort (Educational Baseline)

Insertion sort builds a sorted section at the front of the list one element at a time. Think of sorting playing cards: you pick up each new card and slide it left past any cards that are larger, until it reaches its correct position. At every step, everything to the left of the current position is sorted; the right side is the unsorted remainder yet to be processed.

- Time complexity: $O(n^2)$ average/worst; $O(n)$ best (already sorted — only comparisons, no shifts)
- Space complexity: $O(1)$ extra — fully in-place
- Useful on small or nearly sorted inputs; Python's Timsort uses insertion sort internally for small sub-arrays

In [4]:
def insertion_sort(items):
    for i in range(1, len(items)):
        key = items[i]
        j = i - 1

        while j >= 0 and items[j] > key:
            items[j + 1] = items[j]
            j -= 1

        items[j + 1] = key
    return items


### Bubble Sort (Teaching Baseline)

Bubble sort makes repeated passes through the list, comparing each adjacent pair and swapping them if they are out of order. After each full pass, the largest unsorted element has "bubbled up" to its final position at the end of the unsorted region. An early-exit flag (`swapped`) lets the loop terminate as soon as a full pass completes with no swaps, which means the list is already sorted.

Despite the optimization, bubble sort is $O(n^2)$ on average and worst-case inputs. It is included here for conceptual contrast and complexity discussion, not for practical use.

In [6]:
def bubble_sort(items):
    n = len(items)
    for i in range(n):
        swapped = False
        for j in range(0, n - i - 1):
            if items[j] > items[j + 1]:
                items[j], items[j + 1] = items[j + 1], items[j]
                swapped = True
        if not swapped:
            break
    return items


### Merge Sort

Merge sort follows a **divide-and-conquer** strategy: recursively split the list in half until every sub-list contains one element (a single element is trivially sorted), then repeatedly **merge** pairs of sorted sub-lists back into a single sorted list.

The merge step is where the work happens: two pointers scan the left and right halves simultaneously, always appending the smaller front element to the result. Because each element is visited once per level of recursion, each level costs $O(n)$. The recursion depth is $O(\log n)$ (halving $n$ until size 1), so the total is $O(n \log n)$ — and this holds for every input, regardless of order.

- Time complexity: $O(n \log n)$ best / average / worst
- Space complexity: $O(n)$ extra — each merge allocates a new list
- Stable: equal elements maintain their original relative order

In [5]:
def merge(left, right):
    merged = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged

def merge_sort(items):
    if len(items) <= 1:
        return items[:]

    mid = len(items) // 2
    left = merge_sort(items[:mid])
    right = merge_sort(items[mid:])
    return merge(left, right)

### Quick Sort (Advanced)

Quicksort is also divide-and-conquer, but partitions by value rather than by position. It selects a **pivot** element and rearranges all other values into three groups: elements less than the pivot, elements equal to the pivot, and elements greater. It then recursively sorts the less-than and greater-than groups. No merge step is needed — the three already-sorted pieces concatenate directly into a sorted list.

**Pivot choice determines performance.** With a well-chosen pivot that splits the data roughly in half, recursion depth is $O(\log n)$ and total work is $O(n \log n)$. With a consistently bad pivot — for example, always picking the minimum value on an already-sorted list — one partition is always empty and the other has $n-1$ elements, producing $O(n^2)$ behaviour. **Random pivoting** avoids this: a random element is unlikely to be the extreme value on every level, so expected performance remains $O(n \log n)$ across typical inputs.

- Average: $O(n \log n)$
- Worst-case: $O(n^2)$ — degenerate pivot at every level
- Space: $O(\log n)$ average stack frames from recursion; $O(n)$ worst-case

In [7]:
def quicksort(items):
    if len(items) <= 1:
        return items[:]

    pivot = items[random.randint(0, len(items) - 1)]
    less = [x for x in items if x < pivot]
    equal = [x for x in items if x == pivot]
    greater = [x for x in items if x > pivot]

    return quicksort(less) + equal + quicksort(greater)

### Quick Demo Across Algorithms

All implementations should match Python's built-in sorted output.

In [8]:
demo = [5, 2, 4, 6, 1, 3]
print('insertion:', insertion_sort(demo[:]))  # copy; in-place mutates input
print('merge    :', merge_sort(demo))
print('bubble   :', bubble_sort(demo[:]))     # copy; in-place mutates input
print('quick    :', quicksort(demo))
print('builtin  :', sorted(demo))


insertion: [1, 2, 3, 4, 5, 6]
merge    : [1, 2, 3, 4, 5, 6]
bubble   : [1, 2, 3, 4, 5, 6]
quick    : [1, 2, 3, 4, 5, 6]
builtin  : [1, 2, 3, 4, 5, 6]


## Algorithm Comparison

| Algorithm | Time (avg) | Time (worst) | Space | In-place? | Stable? | Recommended for |
|---|---|---|---|---|---|---|
| Bubble sort | $O(n^2)$ | $O(n^2)$ | $O(1)$ | Yes | Yes | Teaching only |
| Insertion sort | $O(n^2)$ | $O(n^2)$ | $O(1)$ | Yes | Yes | Small or nearly-sorted data |
| Merge sort | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ | No | Yes | Stable sort required; linked lists |
| Quicksort | $O(n \log n)$ | $O(n^2)$ | $O(\log n)$* | No† | No | General-purpose; typically fastest in practice |
| Python `sorted()` / `.sort()` | $O(n \log n)$ | $O(n \log n)$ | $O(n)` / `$O(1)$ | `.sort()` only | Yes | Default choice for all real code |

\* Stack space from recursion. † This implementation returns a new list; classical quicksort swaps in-place.

**Default advice:** always use `sorted()` or `.sort()` in real code. Study the others to understand trade-offs and Big O reasoning.

In [ ]:
### Exercise: Algorithm Planning
#   1. Pick a problem of your choice (for example, "find duplicates in a list").
#   2. Write a 5-step algorithm plan as comments: Input → Process steps → Output → Edge cases → Tests.
### Your code starts here.




### Your code ends here.

In [11]:
### Solution

# Example (find duplicates in a list):
# 1. Input: list of values; Output: list of duplicate values.
# 2. Create an empty set `seen` and empty set `dups`.
# 3. Loop through each value; if in `seen`, add to `dups`; else add to `seen`.
# 4. Convert `dups` to sorted list for consistent output.
# 5. Test edge cases: empty list, no duplicates, all duplicates.

## Correctness and Validation

All four implementations must return the same result as Python's built-in `sorted()` for any valid input. Randomized testing is used rather than hand-picked cases because it exercises orderings and value distributions that manual test design tends to miss — including duplicates, already-sorted inputs, and reverse-sorted inputs across many sizes. Always run correctness checks before timing: a subtle bug that only surfaces on certain inputs could silently corrupt the benchmark results.

In [9]:
for _ in range(50):
    data = [random.randint(0, 1000) for _ in range(60)]
    expected = sorted(data)

    assert insertion_sort(data[:]) == expected
    assert merge_sort(data) == expected
    assert bubble_sort(data[:]) == expected
    assert quicksort(data) == expected

print('All randomized checks passed.')


All randomized checks passed.


## Performance by Input Pattern

Pattern matters. Compare random, sorted, reverse, and duplicate-heavy data.

Timing continuity:
- `time.time()` works for basic measurement.
- `time.perf_counter()` is preferred in these short benchmarks for better precision.

Connection to Chapter 08:
- As before, use timing to support Big O intuition rather than focusing on tiny decimal differences.

In [10]:
# Shared timing utility — see also 1001-searching.ipynb for the 30-repeat version.
def average_runtime(fn, *args, repeats=8):
    total = 0.0
    for _ in range(repeats):
        start = perf_counter()
        fn(*args)
        total += perf_counter() - start
    return total / repeats

n = 1200
random_data = [random.randint(0, 10_000) for _ in range(n)]
sorted_data = sorted(random_data)
reverse_data = sorted_data[::-1]
duplicate_data = [random.choice([1, 2, 3, 4, 5]) for _ in range(n)]

datasets = [
    ('random', random_data),
    ('sorted', sorted_data),
    ('reverse', reverse_data),
    ('duplicates', duplicate_data),
]

print(f"{'pattern':>12} {'bubble':>10} {'insertion':>12} {'merge':>10} {'quick':>10} {'builtin':>10}")
for label, data in datasets:
    t_bubble = average_runtime(lambda d: bubble_sort(d[:]), data)
    t_insert = average_runtime(lambda d: insertion_sort(d[:]), data)
    t_merge = average_runtime(merge_sort, data)
    t_quick = average_runtime(quicksort, data)
    t_builtin = average_runtime(sorted, data)
    print(f"{label:>12} {t_bubble:>10.6f} {t_insert:>12.6f} {t_merge:>10.6f} {t_quick:>10.6f} {t_builtin:>10.6f}")

     pattern     bubble    insertion      merge      quick    builtin


      random   0.030973     0.014517   0.001080   0.001019   0.000037
      sorted   0.000030     0.000071   0.000723   0.001885   0.000003


     reverse   0.037807     0.027260   0.000745   0.000926   0.000004


  duplicates   0.026750     0.011232   0.000938   0.000118   0.000028


In [ ]:
### Exercise: Counting Sort
#   Unlike comparison-based sorts, counting sort runs in O(n + k) where k is the value range.
#   Implement `counting_sort_nonnegative(items)`:
#   1. Accept only nonnegative integers; raise `ValueError` otherwise.
#   2. Return a new sorted list (do not sort in place).
#   3. As a comment, explain when counting sort is preferable to merge sort.
### Your code starts here.




### Your code ends here.

In [12]:
### Solution

def counting_sort_nonnegative(items):
    if not items:
        return []

    max_value = max(items)
    counts = [0] * (max_value + 1)

    for value in items:
        if value < 0:
            raise ValueError('counting_sort_nonnegative only supports nonnegative integers')
        counts[value] += 1

    result = []
    for value, freq in enumerate(counts):
        result.extend([value] * freq)
    return result

exercise_data = [3, 1, 2, 1, 0, 3, 2]
print(counting_sort_nonnegative(exercise_data))

[0, 1, 1, 2, 2, 3, 3]
